**In a large-scale data processing project, multiple PySpark notebooks require the same set of custom transformation functions, such as date formatting, null handling, and data validation. Instead of duplicating the code across notebooks, the data engineering team creates a reusable Python class to centralize these common functions. This approach ensures consistency, reduces maintenance effort, promotes code reusability, and improves overall code readability across the project.**


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
class DataValidation:

    def __init__(self, df):
        self.df = df

    def dedup(self, keyCol, cdcCol):
        df = self.df.withColumn(
            "dedup",
            row_number().over(
                Window.partitionBy(keyCol).orderBy(desc(cdcCol))
            )
        )
        df = df.filter(col('dedup') == 1).drop('dedup')

        return df

    def removeNulls(self, nullCol):
        df = self.df.filter(col(nullCol).isNotNull())
        return df

In [0]:
df = spark.createDataFrame(
    [("1", "2020-01-01", 100),
     ("2", "2020-01-02", 200),
     ("3", "2020-01-03", 300),
     ("1", "2020-01-12", 200)],
    ["order_id", "order_date", "amount"]
)

display(df)

In [0]:
df_dedup = cls_obj.dedup('order_id', 'order_date')

display(df_dedup)